# Wrapping custom atomistic models as representations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/luigibonati/mlcolvar/blob/main/docs/notebooks/tutorials/adv_atomistic_wrapper.ipynb)

**Prerequisites**

- Basic familiarity with graph-based CVs in `mlcolvar`
- Familiarity with PyTorch modules and atomistic graph inputs

**Author:** Kai Zhu

## Overview

This tutorial focuses on one question:

> **How can a custom atomistic model be connected to the common `mlcolvar`
> representation interface?**

If an adapter already exists for the atomistic model, such as
`MACERepresentation`, see the companion tutorial
**Using pretrained atomistic models as representations**.

Here we instead build a minimal wrapper from scratch:

```text
mlcolvar graph dictionary
        ↓
custom Representation
(input_kind="graph")
        ↓
atomistic model
        ↓
atom- or system-level latent representation
        ↓
downstream CV
```

`Representation` provides a common interface for both descriptor- and
graph-based inputs. For an atomistic model, the wrapper sets
`input_kind="graph"` and provides the graph-specific metadata required for
dataset construction, pooling, and downstream `mlcolvar` workflows.


## 1. The `Representation` interface for graph inputs

A custom atomistic wrapper should subclass `Representation` and set
`input_kind="graph"`. The wrapper has two main responsibilities:

1. declare the metadata required to construct compatible atomistic graphs;
2. translate the standard `mlcolvar` graph dictionary into the inputs expected
   by the external model and return its latent features.

The most important arguments for graph inputs are:

| Argument | Meaning |
| --- | --- |
| `out_features` | Number of latent features returned per atom or system |
| `input_kind` | Set to `"graph"` for atomistic graph inputs |
| `atomic_numbers` | Atomic species supported by the model |
| `cutoff` | Neighbor-list cutoff used by the model |
| `pooling_operation` | Optional `"mean"` or `"sum"` atom-to-system reduction |
| `buffer` | Optional buffer added to the graph cutoff |
| `long_range_cutoff` | Optional long-range interaction cutoff |
| `freeze` | Whether the wrapped model parameters remain fixed |

A graph dictionary can contain fields such as `positions`, `node_attrs`,
`edge_index`, `shifts`, `unit_shifts`, `batch`, `ptr`, `cell`, and `pbc`.
The wrapper only needs to consume the fields required by the external model.

Task-specific CV heads should remain outside the representation.


## 2. Define a minimal atomistic backend

The toy model below mimics a simple message-passing network. It embeds atomic
species, constructs distance-dependent messages along graph edges, and returns
one latent vector per atom.

The model is deliberately simple and is **not** intended to be a physically
meaningful atomistic architecture. Its only purpose is to expose the inputs,
outputs, and gradient behavior required by the wrapper interface.

In a real application, this class would be replaced by a pretrained or
user-defined atomistic model.


In [2]:
from collections.abc import Sequence
from typing import Optional

import torch
from torch import nn
from torch_geometric.data import Data

from mlcolvar.representation import Representation, evaluate_dataset

torch.manual_seed(0)
torch.set_default_dtype(torch.float32)


class ToyAtomisticModel(nn.Module):
    """Minimal atomistic backend used to demonstrate the wrapper interface."""

    def __init__(self, n_species: int, hidden_features: int) -> None:
        super().__init__()
        self.embedding = nn.Linear(n_species, hidden_features, bias=False)

    def forward(
        self,
        positions: torch.Tensor,
        node_attrs: torch.Tensor,
        edge_index: torch.Tensor,
        shifts: torch.Tensor,
    ) -> torch.Tensor:
        node_features = self.embedding(node_attrs)
        sender, receiver = edge_index

        vectors = positions[receiver] - positions[sender] + shifts
        distances = torch.linalg.norm(vectors, dim=-1, keepdim=True)
        messages = node_features[sender] * torch.exp(-distances)

        aggregated = torch.zeros_like(node_features)
        aggregated.index_add_(0, receiver, messages)
        return node_features + aggregated


## 3. Implement the wrapper

The wrapper declares the representation metadata in `__init__()` and performs
model-specific feature extraction in `forward()`.

The toy backend returns atom-level features. Since this wrapper consumes
atomistic graphs, `input_kind="graph"` is passed to the common
`Representation` base class. Calling `self.pooling(...)` keeps the features
atom-level when `pooling_operation=None`, or reduces them to one vector per
system when `"mean"` or `"sum"` is selected.

A particularly important distinction is that

```text
freeze=True
    ├─ backend parameter gradients: disabled
    └─ coordinate gradients:        still available
```

This allows a frozen atomistic representation to be differentiated with
respect to coordinates when the downstream CV or simulation workflow requires
forces or coordinate derivatives.


In [3]:
class ToyRepresentation(Representation):
    """Adapt ToyAtomisticModel to the common mlcolvar representation interface."""

    def __init__(
        self,
        model: nn.Module,
        *,
        hidden_features: int,
        atomic_numbers: Sequence[int],
        cutoff: float,
        pooling_operation: Optional[str] = None,
        freeze: bool = True,
    ) -> None:
        super().__init__(
            out_features=hidden_features,
            input_kind="graph",
            atomic_numbers=atomic_numbers,
            cutoff=cutoff,
            pooling_operation=pooling_operation,
            freeze=freeze,
        )
        self.model = model
        self._freeze_module(self.model)

    def forward(
        self,
        data: dict[str, torch.Tensor],
        cell: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        del cell

        features = self.model(
            positions=data["positions"],
            node_attrs=data["node_attrs"],
            edge_index=data["edge_index"],
            shifts=data["shifts"],
        )
        return self.pooling(features, data)


## 4. Sanity-check the wrapper

The following three-atom graph uses two atomic species. The columns of
`node_attrs` follow the same ordering as `atomic_numbers=[1, 8]`.

We first keep the representation atom-level and verify three interface
properties:

- the output shape is `(n_atoms, out_features)`;
- the frozen backend has no trainable parameters;
- coordinate gradients remain available.


In [ ]:
toy_backend = ToyAtomisticModel(n_species=2, hidden_features=8)
atom_representation = ToyRepresentation(
    model=toy_backend,
    hidden_features=8,
    atomic_numbers=[1, 8],
    cutoff=5.0,
    freeze=True,
)

example_graph = {
    "positions": torch.tensor(
        [
            [0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
        ],
        requires_grad=True,
    ),
    "node_attrs": torch.tensor(
        [
            [1.0, 0.0],
            [0.0, 1.0],
            [1.0, 0.0],
        ]
    ),
    "edge_index": torch.tensor(
        [
            [0, 1, 0, 2, 1, 2],
            [1, 0, 2, 0, 2, 1],
        ],
        dtype=torch.long,
    ),
    "shifts": torch.zeros(6, 3),
    "batch": torch.zeros(3, dtype=torch.long),
    "ptr": torch.tensor([0, 3], dtype=torch.long),
}

atom_features = atom_representation(example_graph)
coordinate_gradient = torch.autograd.grad(
    atom_features.sum(),
    example_graph["positions"],
)[0]

assert atom_features.shape == (3, 8)
assert not any(
    parameter.requires_grad
    for parameter in atom_representation.model.parameters()
)
assert coordinate_gradient.shape == example_graph["positions"].shape

print("Atom-level output shape:", tuple(atom_features.shape))
print("Coordinate-gradient shape:", tuple(coordinate_gradient.shape))


Output kind: atom
Atom-level output shape: (3, 8)
Coordinate-gradient shape: (3, 3)


### System-level output

Most downstream CV models expect one vector per molecular configuration. The
same wrapper can provide this directly through `pooling_operation`.


In [ ]:
representation = ToyRepresentation(
    model=toy_backend,
    hidden_features=8,
    atomic_numbers=[1, 8],
    cutoff=5.0,
    pooling_operation="mean",
    freeze=True,
)

system_features = representation(example_graph)

assert system_features.shape == (1, 8)

print("System-level output shape:", tuple(system_features.shape))


Output kind: system
System-level output shape: (1, 8)


## 5. Use the custom representation in a CV

The purpose of this section is only to verify that the wrapper participates in
a normal `mlcolvar` workflow. We therefore use a small synthetic two-state
dataset rather than turning this notebook into a second DeepTDA tutorial.

The two states differ mainly in their interatomic distances. This changes the
distance-dependent messages produced by the toy backend and therefore its
latent representation.

The samples are interleaved as `A, B, A, B, ...` so that even-sized sequential
batches contain both states.


In [6]:
from mlcolvar.data import DictDataset

edge_index = torch.tensor(
    [
        [0, 1, 0, 2, 1, 2],
        [1, 0, 2, 0, 2, 1],
    ],
    dtype=torch.long,
)

node_attrs = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 0.0],
    ]
)


def make_graph(distance: float) -> Data:
    positions = torch.tensor(
        [
            [0.0, 0.0, 0.0],
            [distance, 0.0, 0.0],
            [0.0, distance, 0.0],
        ]
    )
    return Data(
        positions=positions,
        node_attrs=node_attrs.clone(),
        edge_index=edge_index.clone(),
        shifts=torch.zeros(edge_index.shape[1], 3),
    )


n_per_state = 200
distances_a = 0.9 + 0.08 * torch.randn(n_per_state)
distances_b = 1.8 + 0.08 * torch.randn(n_per_state)

graphs = []
labels = []
for distance_a, distance_b in zip(distances_a, distances_b):
    graphs.extend(
        [
            make_graph(float(distance_a)),
            make_graph(float(distance_b)),
        ]
    )
    labels.extend([0.0, 1.0])

labels = torch.tensor(labels)

dataset = DictDataset(
    dictionary={
        "data_list": graphs,
        "labels": labels,
    },
    metadata={
        "atomic_numbers": [1, 8],
        "cutoff": 5.0,
        "buffer": 0.0,
        "long_range_cutoff": -1.0,
    },
    data_type="graphs",
)

dataset = representation.align_dataset(dataset)

print("Number of samples:", len(dataset))
print("State A samples:", int((labels == 0).sum()))
print("State B samples:", int((labels == 1).sum()))


Number of samples: 400
State A samples: 200
State B samples: 200


### 5.1 Precompute the representation and train a downstream CV

Because the representation is frozen, its system-level features can be
evaluated once before training. The remaining model is a standard DeepTDA CV
with a small `FeedForward` head.

This is intentionally a compact integration test: the focus of the tutorial is
the representation interface rather than DeepTDA optimization.


In [ ]:
from lightning import Trainer

from mlcolvar.core import FeedForward
from mlcolvar.cvs import DeepTDA
from mlcolvar.data import DictModule

features = evaluate_dataset(
    representation,
    dataset,
    batch_size=64,
)

feature_dataset = DictDataset(
    {
        "data": features,
        "labels": labels,
    }
)

datamodule = DictModule(
    feature_dataset,
    batch_size=32,
    lengths=[0.8, 0.2],
    random_split=False,
    shuffle=False,
)

head = FeedForward([representation.out_features, 32, 32, 1])
model = DeepTDA(
    n_states=2,
    n_cvs=1,
    target_centers=[-7.0, 7.0],
    target_sigmas=[0.2, 0.2],
    model=head,
)

trainer = Trainer(
    max_epochs=5,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)
trainer.fit(model, datamodule)

print("Feature shape:", tuple(features.shape))


/home/kzhu-iit.local/data/mlcolvar/mlcolvar/data/datamodule.py:136: UserWarning: A torch.generator was provided but it is not used with random_split=False
  warnings.warn(
/home/kzhu-iit.local/miniconda3/envs/mlcolvar-gnn/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 99: 100%|██████████| 10/10 [00:00<00:00, 156.83it/s]                 

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 10/10 [00:00<00:00, 155.12it/s]
Feature shape: (400, 8)


### 5.2 Verify raw-graph inference

Training used precomputed representation features. For inference on raw
atomistic graphs, attach the frozen representation as the CV `preprocessing`
module and verify that both inference paths agree.


In [8]:
import copy

raw_model = copy.deepcopy(model).eval()
raw_model.preprocessing = representation

raw_predictions = evaluate_dataset(
    raw_model,
    dataset,
    batch_size=64,
).reshape(-1)

with torch.no_grad():
    feature_predictions = model(features).reshape(-1)

torch.testing.assert_close(
    raw_predictions,
    feature_predictions,
    rtol=1e-4,
    atol=1e-5,
)

print("Precomputed-feature and raw-graph predictions agree.")


Precomputed-feature and raw-graph predictions agree.


## 6. Adapting the wrapper to a real atomistic model

The toy wrapper above is intentionally minimal. When adapting a real atomistic
model, the same interface is used, but the wrapper may need to translate
additional backend-specific conventions.

| Component | What the wrapper should handle |
| --- | --- |
| Atomic species | Match `node_attrs` to the species convention of the backend |
| Coordinates | Convert length units when required |
| Neighbor list | Translate `edge_index` and shifts, or rebuild the list if required |
| Model input | Convert the `mlcolvar` graph dictionary into backend-specific inputs |
| Model output | Extract the desired latent representation from the backend |
| Pooling | Return atom-level features or reduce them to system-level features |
| Freezing | Freeze model parameters while preserving coordinate derivatives |

The representation should contain only reusable atomistic feature extraction.
Task-specific heads and CV objectives should remain downstream.
